In [1]:
import math
from collections import defaultdict, Counter
from MCTS_cross_encoder_batch import RLEnhancedGraphRAG, main
from langchain_neo4j import Neo4jGraph
from sentence_transformers import SentenceTransformer
import json
import os
# Add semantic retriever import
from semantic_retriever import main as semantic_main



/home/ec2-user/graphrag_evaluation/graphrag_venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ec2-user/graphrag_evaluation/graphrag_venv/lib64/python3.9/site-packages/networkx/utils/backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


In [2]:
def load_jsonl(file_path):
    """Load a JSONL file and return a list of dictionaries"""
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data


In [3]:
data=load_jsonl('/home/ec2-user/repoqa-2024-06-23.json')

In [4]:
def path_to_module_name(path):
    """
    Convert file path to module name.
    Example: 'src/package/module.py' -> 'src.package.module'
    """
    # Remove .py extension
    if path.endswith(".py"):
        path = path[:-3]

    # Convert path separators to dots
    module_name = path.replace("/", ".").replace("\\", ".")

    # Handle __init__.py files - remove the __init__ part
    if module_name.endswith(".__init__"):
        module_name = module_name[:-9]

    return module_name

In [5]:
def extract_purpose(text):
    lines = text.split('\n')
    for line in lines:
        if 'Purpose' in line:
            # Find the position of 'Purpose' and extract everything after it
            purpose_idx = line.find('Purpose')
            if purpose_idx != -1:
                # Look for ':' after 'Purpose'
                colon_idx = line.find(':', purpose_idx)
                if colon_idx != -1:
                    return line[colon_idx + 1:].strip()
                else:
                    # If no colon, take everything after 'Purpose'
                    return line[purpose_idx + 7:].strip()  # 7 is len('Purpose')
    return ""

In [6]:
def evaluate_retrieval_mcts(needle_data, rl_graphrag, repo_name):
    """Evaluation function for MCTS retriever adapted for RepoQA needles"""
    
    description = needle_data['description']
    target_function_name = needle_data['name']
    target_file_path = needle_data['path']
    purpose=extract_purpose(description)
    # Get search results from MCTS
    res = rl_graphrag.search(purpose, repo_name=repo_name)
    
    # Store match paths for this query
    match_paths = []
    
    print(f"\nSearching for function '{target_function_name}' in '{target_file_path}'")
    print(f"Query: {description}")
    print(f"Found {len(res)} high-reward nodes:")
    for i, node_info in enumerate(res, 1):
        print(
            f"{i}. {node_info['node_data'].get('name', 'Unknown')} "
            f"(Reward: {node_info['avg_reward']:.2f}, "
            f"Visits: {node_info['visit_count']})"
        )
    
    # Convert target file path to module name for comparison
    target_module_name = path_to_module_name(target_file_path)
    
    # Extract names from retrieved results and check for matches
    retrieved_relevances = []
    found_match = False
    
    for i in range(min(10, len(res))):  # Only consider top 10
        relevance = 0
        node_data = res[i]['node_data']
        tree_node = res[i].get('tree_node')  # Get the TreeNode object
        
        # Extract the name based on node type
        retrieved_function_name = node_data.get('name', '')
        retrieved_module_name = node_data.get('module_name', '')
        
        # If the retrieved node is a Class, expand its methods and evaluate against the target
        try:
            node_type = node_data.get('node_type')
            labels = node_data.get('labels') or []
            if tree_node and not node_type:
                node_type = getattr(tree_node, 'graph_node_data', {}).get('node_type')
            if tree_node and not labels:
                labels = getattr(tree_node, 'graph_node_data', {}).get('labels') or []
        except Exception:
            node_type = None
            labels = []
        
        is_class_node = (node_type == 'Class') or (isinstance(labels, list) and 'Class' in labels)
        if is_class_node and tree_node and hasattr(tree_node, 'graph_node_id'):
            methods = []
            graph_obj = getattr(rl_graphrag, 'graph', None)
            if graph_obj:
                try:
                    methods = graph_obj.query(
                        """
                        MATCH (c) WHERE c.name = $name
                        MATCH (c)-[:HAS_METHOD]-(m:Method)
                        RETURN m.name AS name, m.module_name AS module_name, c.name AS class_name
                        """,
                        params={"name": tree_node.graph_node_data.get('name')}
                    )
                except Exception as e:
                    print(f"  Warning: failed to expand class methods: {e}")
            
            for m in methods:
                method_name = m.get('name', '')
                method_module = m.get('module_name', '')
                method_class = m.get('class_name', '')
                class_match = False
                
                if '.' in target_function_name:
                    target_class, target_method = target_function_name.rsplit('.', 1)
                    if (
                        method_name == target_method
                        and method_class == target_class
                        and (not method_module or method_module == target_module_name)
                    ):
                        class_match = True
                else:
                    if method_name == target_function_name and (not method_module or method_module == target_module_name):
                        class_match = True
                
                if class_match:
                    relevance = 1
                    found_match = True
                    if tree_node:
                        match_info = {
                            'description': description,
                            'target_function': target_function_name,
                            'target_file': target_file_path,
                            'target_module': target_module_name,
                            'repo_name': repo_name,
                            'retrieved_function': f"{method_class}.{method_name}",
                            'retrieved_module': method_module,
                            'relevance': relevance,
                            'rank': i + 1,
                        }
                        match_paths.append(match_info)
                        print(f"CLASS CONTAINS MATCH: {target_function_name} at rank {i+1}")
                    # Do not break outer loop; we still append relevance for this rank below
                    break
        
        # Check for exact function name match
        if retrieved_function_name == target_function_name:
            # Also check if it's in the correct module (if module info is available)
            if not retrieved_module_name or retrieved_module_name == target_module_name:
                relevance = 1  # Binary relevance for RepoQA
                found_match = True
                
                # Store the path for this match
                if tree_node:
                    # path = get_node_path_to_root(tree_node)
                    match_info = {
                        'description': description,
                        'target_function': target_function_name,
                        'target_file': target_file_path,
                        'target_module': target_module_name,
                        'repo_name': repo_name,
                        'retrieved_function': retrieved_function_name,
                        'retrieved_module': retrieved_module_name,
                        'relevance': relevance,
                        'rank': i + 1,
                        # 'path': path
                    }
                    match_paths.append(match_info)
                    print(f"EXACT MATCH FOUND: {target_function_name} at rank {i+1}")
        
        # Also check for method matches (class.method pattern)
        elif '.' in target_function_name:
            class_name, method_name = target_function_name.rsplit('.', 1)
            if retrieved_function_name == method_name:
                # Check if this method belongs to the right class
                retrieved_class = node_data.get('class', '')
                if retrieved_class == class_name:
                    relevance = 1
                    found_match = True
                    
                    # Store the path for this method match
                    if tree_node:
                        # path = get_node_path_to_root(tree_node)
                        match_info = {
                            'description': description,
                            'target_function': target_function_name,
                            'target_file': target_file_path,
                            'target_module': target_module_name,
                            'repo_name': repo_name,
                            'retrieved_function': f"{retrieved_class}.{retrieved_function_name}",
                            'retrieved_module': retrieved_module_name,
                            'relevance': relevance,
                            'rank': i + 1,
                            'path': path
                        }
                        match_paths.append(match_info)
                        print(f"METHOD MATCH FOUND: {target_function_name} at rank {i+1}")
        
        retrieved_relevances.append(relevance)
    
    # Store the match paths for this query
    if match_paths:
        safe_repo_name = repo_name.replace('/', '_').replace(' ', '_')
        # store_match_paths(match_paths, filename=f"traces/repoqa_match_paths_{safe_repo_name}.json")
    
    # Pad with zeros if we have less than 10 results
    while len(retrieved_relevances) < 10:
        retrieved_relevances.append(0)
    
    # Calculate metrics
    def calculate_dcg(relevances):
        dcg = 0.0
        for i, rel in enumerate(relevances):
            if rel > 0:
                dcg += rel / math.log2(i + 2)  # i+2 because positions are 1-indexed
        return dcg
    
    # DCG for retrieved results
    dcg = calculate_dcg(retrieved_relevances)
    
    # IDCG - ideal DCG (1 relevant item at position 1)
    ideal_relevances = [1] + [0] * 9  # Only one relevant item per needle
    idcg = calculate_dcg(ideal_relevances)
    
    # NDCG@10
    ndcg_10 = dcg / idcg if idcg > 0 else 0.0
    
    # Calculate Recall@10 (binary: either we found it or we didn't)
    recall_10 = 1.0 if found_match else 0.0
    
    return {
        'ndcg_10': ndcg_10,
        'recall_10': recall_10,
        'total_relevant': 1,  # Always 1 for needle tasks
        'relevant_retrieved': 1 if found_match else 0,
        'target_function': target_function_name,
        'target_file': target_file_path,
        'found_match': found_match
    }


In [7]:
graph_config = {
        "url": "bolt://localhost:7687",
        "username": "neo4j",
        "password": "nutanix_neo4j",
    }
    
graph = Neo4jGraph(**graph_config)
embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: mixedbread-ai/mxbai-embed-large-v1
INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'passage']


In [8]:
rl_graphrag = RLEnhancedGraphRAG(
            graph=graph,
            embedding_model=embedding_model,
            max_iterations=200,
            reward_threshold=5.0,
            alpha=0.9,
            cross_encoder_model="BAAI/bge-reranker-v2-m3",
            top_k_children=80,
            top_k_references=10,
            reduce_top_k_flag=True,
            min_top_k_children=20,
            exploration_param=0.25
        )

INFO:MCTS_cross_encoder_batch:Loading cross-encoder model: BAAI/bge-reranker-v2-m3
INFO:MCTS_cross_encoder_batch:Cross-encoder model loaded successfully
INFO:MCTS_cross_encoder_batch:Loading tokenizer for cross-encoder model: BAAI/bge-reranker-v2-m3
INFO:MCTS_cross_encoder_batch:Tokenizer loaded successfully


In [9]:
base_clone_dir = "/home/ec2-user/RepoQA/"

In [29]:
repo_path_raw=data[0]['python'][8]['repo']
print(repo_path_raw)

ethereum/web3.py


In [30]:
if repo_path_raw.startswith('https://'):
    repo_url = repo_path_raw
else:
    repo_url = f"https://github.com/{repo_path_raw}.git"

repo_name = repo_url.split('/')[-1].replace('.git', '')
if repo_url.count('/') >= 4:
    owner = repo_url.split('/')[-2]
    repo_name = f"{owner}_{repo_name}"

repo_path = os.path.join(base_clone_dir, repo_name)
print(repo_path)

/home/ec2-user/RepoQA/ethereum_web3.py


In [31]:
module_count_query = f"""
MATCH (r:Repo {{name:'{repo_path}'}})-[:CONTAINS]->(m:Module)
RETURN count(m) as module_count
"""
module_result = graph.query(module_count_query)
module_count = module_result[0]['module_count'] if module_result else 100

if module_count < 20:
    rl_graphrag.original_top_k_children = max(1, module_count // 2)
else:
    rl_graphrag.original_top_k_children = min(round(module_count / 10) * 5, 200) + 15

rl_graphrag.exploration_param = 1 / (1 * math.sqrt(math.log(4 * rl_graphrag.original_top_k_children)))
print(f"MCTS exploration_param: {rl_graphrag.exploration_param}")

MCTS exploration_param: 0.414195139525592


In [13]:
rl_graphrag.original_top_k_children

40

In [39]:
print(data[0]['python'][8]['needles'][8])

{'name': '_make_get_request', 'start_line': 64, 'end_line': 67, 'start_byte': 1317, 'end_byte': 1502, 'global_start_line': 27007, 'global_end_line': 27010, 'global_start_byte': 856733, 'global_end_byte': 856918, 'code_ratio': 0.0, 'path': 'web3/beacon/beacon.py', 'description': '\n1. **Purpose**: The function is designed to fetch data from a specified server endpoint and return the response in a structured format.\n2. **Input**: It takes a string representing the specific endpoint of a server from which data needs to be retrieved.\n3. **Output**: It returns a dictionary containing the data fetched from the server endpoint.\n4. **Procedure**: The function constructs the full URL by appending the endpoint to the base server URL. It then sends an HTTP GET request to this URL and waits for the response. The response is processed and returned as a JSON-formatted dictionary.\n'}


In [40]:
print(data[0]['python'][8]['needles'][8]['description'])


1. **Purpose**: The function is designed to fetch data from a specified server endpoint and return the response in a structured format.
2. **Input**: It takes a string representing the specific endpoint of a server from which data needs to be retrieved.
3. **Output**: It returns a dictionary containing the data fetched from the server endpoint.
4. **Procedure**: The function constructs the full URL by appending the endpoint to the base server URL. It then sends an HTTP GET request to this URL and waits for the response. The response is processed and returned as a JSON-formatted dictionary.



In [53]:
eval_result = evaluate_retrieval_mcts(data[0]['python'][8]['needles'][8], rl_graphrag, repo_path)

INFO:MCTS_cross_encoder_batch:[MCTS] Starting search for query: 'The function is designed to fetch data from a specified server endpoint and return the response in a structured format.' in repo: '/home/ec2-user/RepoQA/ethereum_web3.py'
Batches: 100%|██████████| 1/1 [00:00<00:00, 77.98it/s]
INFO:MCTS_cross_encoder_batch:Query embedding created!
INFO:MCTS_cross_encoder_batch:[MCTS] Initialized tree with root: Repo:/home/ec2-user/RepoQA/ethereum_web3.py
INFO:MCTS_cross_encoder_batch:[MCTS] Starting 200 iterations with reward_threshold=5.0
INFO:MCTS_cross_encoder_batch:[MCTS] ===== Iteration 1/200 =====
INFO:MCTS_cross_encoder_batch:[MCTS] Phase 1 - Selected: Repo:/home/ec2-user/RepoQA/ethereum_web3.py (visits: 0, avg_reward: 0.00)
INFO:MCTS_cross_encoder_batch:[MCTS] Phase 2 - Expanding tree leaf: Repo:/home/ec2-user/RepoQA/ethereum_web3.py
INFO:MCTS_cross_encoder_batch:[MCTS] Phase 2 - Found 141 children, evaluating similarity
INFO:MCTS_cross_encoder_batch:[MCTS] Phase 2 - First root exp


Searching for function '_make_get_request' in 'web3/beacon/beacon.py'
Query: 
1. **Purpose**: The function is designed to fetch data from a specified server endpoint and return the response in a structured format.
2. **Input**: It takes a string representing the specific endpoint of a server from which data needs to be retrieved.
3. **Output**: It returns a dictionary containing the data fetched from the server endpoint.
4. **Procedure**: The function constructs the full URL by appending the endpoint to the base server URL. It then sends an HTTP GET request to this URL and waits for the response. The response is processed and returned as a JSON-formatted dictionary.

Found 10 high-reward nodes:
1. get_light_client_finality_update (Reward: 9.38, Visits: 3)
2. _make_get_request (Reward: 9.38, Visits: 2)
3. get_light_client_optimistic_update (Reward: 9.30, Visits: 3)
4. get_light_client_bootstrap_structure (Reward: 9.28, Visits: 3)
5. AsyncMakeRequestFn (Reward: 9.28, Visits: 1)
6. get_f

In [31]:
eval_result

{'ndcg_10': 0.0,
 'recall_10': 0.0,
 'total_relevant': 1,
 'relevant_retrieved': 0,
 'target_function': 'fused_decode4_matmul5_int3_int16_fp16_before',
 'target_file': 'mlc_llm/dispatch/llama/main.py',
 'found_match': False}

In [52]:
queue = [rl_graphrag.root_tree_node]

# Print header
print(f"{'Node ID':<120} {'Total_Reward':<12} {'Visits':<8} {'Sim_Reward':<12} {'Sim_Visits':<10} {'Child_Reward':<12} {'Child_Visits':<10} {'Retrieval_Score':<12} {'Cosine_Similarity':<12}")
print("-" * 100)
i=0
while queue:
    i+=1
    node = queue.pop(0)
    print(f"{node.graph_node_id:<120} {node.total_reward:<12.2f} {node.visit_count:<8} {node.simulation_reward:<12.2f} {node.simulation_visits:<10} {node.children_reward:<12.2f} {node.children_visits:<10} {node.retrieval_score:<12.2f} {node.cosine_similarity}")
    queue.extend(node.children)

Node ID                                                                                                                  Total_Reward Visits   Sim_Reward   Sim_Visits Child_Reward Child_Visits Retrieval_Score Cosine_Similarity
----------------------------------------------------------------------------------------------------
Repo:/home/ec2-user/RepoQA/ethereum_web3.py                                                                              743.64       350      0.00         1          743.64       350        0.00         0.0
Module:web3._utils.caching                                                                                               13.72        11       4.03         1          9.69         10         4.37         0.7378656312918828
Module:web3.utils.async_exception_handling                                                                               0.58         1        0.58         1          0.00         0          1.24         0.7182028812356216
Module:web3.manage

In [18]:
print(i)

824


In [24]:
from FlagEmbedding import FlagReranker
cross_encoder = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=False)


In [17]:
# des="""
# description: **PURPOSE**
# This function is used to get the version control system (VCS) properties from a given source path.

# **IMPLEMENTATION**
# The function uses the `Git.info` method from the `poetry.vcs.git` module to retrieve the VCS information from the given source path. The function then returns a tuple containing the VCS type ("git"), the origin URL, and the revision hash.

# **KEY FEATURES**
# - The function uses the `Git.info` method from the `poetry.vcs.git` module to retrieve the VCS information.
# - The function takes a `Path` object as input and returns a tuple of three strings: the VCS type, the origin URL, and the revision hash.
# - The function is designed to handle any `Path` object that is a valid path to a Git repository.
# """

In [18]:
query=data[0]['python'][1]['needles'][1]['description']
print(query)


1. **Purpose**: To retrieve version control system (VCS) details for a given package directory, specifically identifying the type of VCS, its remote origin URL, and the current revision hash.
2. **Input**: A path to the directory where the package is located.
3. **Output**: A tuple containing three strings: the type of VCS (e.g., 'git'), the URL of the remote origin, and the current revision hash.
4. **Procedure**: The function utilizes a VCS utility to extract information from the specified directory. It first checks the VCS type and then gathers the remote origin URL and the current revision hash from the VCS metadata associated with the directory.



In [19]:
cross_encoder.compute_score((query,des), normalize=True)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[0.04832711107954501]

In [34]:
embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")
embedding_model.to('cuda') 

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: mixedbread-ai/mxbai-embed-large-v1
INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'passage']


SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [45]:
res=graph.query(
    """
   MATCH (m{name: "_make_get_request"})
RETURN m.description, m.member_descriptions
    """,
)

In [46]:
es=embedding_model.encode(res[0]['m.description'] + res[0]['m.member_descriptions'])

Batches: 100%|██████████| 1/1 [00:00<00:00, 38.75it/s]


In [47]:
res=graph.query(
    """
   MATCH (m{name: "_make_get_request"})
   set m.embedding = $embedding
    """, {
        "embedding": es
    }
)